In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 확률적 오류 증폭을 사용한 유틸리티 규모의 오류 완화

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*사용 시간 추정: Heron r3 프로세서에서 14분 (참고: 이는 추정값입니다. 실제 실행 시간은 다를 수 있습니다.)*
## 학습 결과
이 튜토리얼을 마친 후 사용자들은 다음 내용을 이해할 수 있어야 합니다:
- *영잡음 외삽* (ZNE)의 이론, 잡음을 증폭하는 다양한 방법, 그리고 유틸리티 규모의 실험에서 *확률적 오류 증폭* (PEA)이 선호되는 이유.
- Qiskit을 사용하여 PEA를 활용한 ZNE를 실제로 구현하는 방법.

## 사전 요구 사항
이 튜토리얼을 진행하기 전에 다음 주제에 익숙하기를 권장합니다:
- Qiskit에서 오류 완화를 사용하는 기본 지식을 위한 *유틸리티 규모 양자 컴퓨팅* 과정의 [오류 완화 레슨](/learning/courses/utility-scale-quantum-computing/error-mitigation).
- 이 튜토리얼의 예제로 사용된 유틸리티 규모 실험에 대한 추가 배경 지식을 위한 *유틸리티 규모 양자 컴퓨팅* 과정의 [유틸리티-I 레슨](/learning/courses/utility-scale-quantum-computing/utility-i).
## 배경
이 튜토리얼은 *영잡음 외삽* (ZNE)의 실험적 버전인 *확률적 오류 증폭* (PEA)을 사용하여 Qiskit Runtime으로 유틸리티 규모의 오류 완화 실험을 실행하는 방법을 설명합니다.

![kim\_nature\_fig.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1e67c34-9d4d-4a88-9340-f0b2f3676770.avif)
**참고 문헌**: Y. Kim et al. *양자 컴퓨팅의 내결함성 이전 유용성에 대한 증거.* [Nature 618.7965 (2023)](https://www.nature.com/articles/s41586-023-06096-3)
### 영잡음 외삽 (ZNE)
영잡음 외삽 (ZNE)은 *알 수 없는* 잡음의 영향을 제거하는 오류 완화 기법으로, Circuit 실행 중 잡음을 *알려진* 방식으로 조정할 수 있습니다.

기댓값이 잡음에 따라 알려진 함수로 변한다고 가정합니다.

$$
\langle A(\lambda) \rangle = \langle A(0) \rangle + \sum_{k=0}^{m} a_k \lambda^k + R
$$

여기서 $\lambda$는 잡음 강도를 매개변수화하며 증폭할 수 있습니다.

ZNE는 다음 단계로 구현할 수 있습니다.

1. 여러 잡음 인자 $\lambda_1, \lambda_2, ... $에 대해 Circuit 잡음을 증폭합니다.
2. 잡음이 증폭된 모든 Circuit을 실행하여 $\langle A(\lambda_1)\rangle, ...$을 측정합니다.
3. 영잡음 한계 $\langle A(0)\rangle$으로 외삽합니다.

![zne\_stages.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/5e63d706-82d8-4212-b802-c9191ce53341.avif)
#### ZNE를 위한 잡음 증폭
ZNE를 성공적으로 구현하는 데 있어 주요 과제는 기댓값의 잡음에 대한 정확한 모델을 갖추고 잡음을 알려진 방식으로 증폭하는 것입니다.

ZNE에서 오류 증폭을 구현하는 세 가지 일반적인 방법이 있습니다.

| **펄스 스트레칭**                                                                                                                                                                                     | **게이트 폴딩**                                                                                                                                                                                    | **확률적 오류 증폭**                                                                                                                                                                     |
| ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 보정을 통해 펄스 지속 시간 조정                                                                                                                                                                        | 항등 사이클 $U\mapsto U(U^{-1}U)^{\lambda-1}/2$ 에서 Gate 반복                                                                                                                                     | 파울리 채널 샘플링을 통해 잡음 추가                                                                                                                                                       |
| ![zne\_pulse\_stretching.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/83188b57-e88f-43a1-a7bd-29327f46ecf5.avif) | ![zne\_gate\_folding.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1358d08-2632-4fd2-bf0f-f9384a2d3340.avif) | ![zne\_pea.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/3d69d5bd-70e5-4eeb-aa02-fc0a62043010.avif) |
| Kandala et al. Nature (2019)                                                                                                                                                                          | Shultz et al. PRA (2022)                                                                                                                                                                           | Li & Benjamin PRX (2017)                                                                                                                                                                 |
유틸리티 규모 실험에서는 *확률적 오류 증폭* (PEA)이 가장 매력적입니다.

* 펄스 스트레칭은 게이트 잡음이 지속 시간에 비례한다고 가정하는데, 이는 일반적으로 사실이 아닙니다. 보정 비용도 큽니다.
* 게이트 폴딩은 큰 스트레치 인자가 필요하여 실행 가능한 Circuit 깊이를 크게 제한합니다.
* PEA는 기본 잡음 인자 ($\lambda=1$)로 실행 가능한 모든 Circuit에 적용할 수 있지만, 잡음 모델을 학습해야 합니다.
### PEA를 위한 잡음 모델 학습
PEA는 *확률적 오류 취소* (PEC)와 동일한 레이어 기반 잡음 모델을 가정합니다. 다만, Circuit 잡음에 따라 지수적으로 증가하는 샘플링 오버헤드를 피할 수 있습니다.

| **단계 1**                                                                                                                                                                                         | **단계 2**                                                                                                                                                                                        | **단계 3**                                                                                                                                                                                          |
| -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 2-Qubit Gate 레이어를 파울리 트월링                                                                                                                                                                 | 레이어의 항등 쌍을 반복하고 잡음 학습                                                                                                                                                              | 충실도 도출 (각 잡음 채널의 오류)                                                                                                                                                                    |
| ![pec\_pauli\_twirling.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/2eab5ff4-40fa-4a41-9f2c-74f5e22c4643.avif) | ![pec\_learn\_layer.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/8d0d64c3-65ad-4419-8ac9-4ec9633d39a0.avif) | ![pec\_curve\_fitting.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/c51bd42d-2463-4c78-807b-d284ca79296f.avif) |

**참고 문헌**: E. van den Berg, Z. Minev, A. Kandala, and K. Temme, *희소 파울리-린드블라드 모델을 사용한 잡음 양자 프로세서에서의 확률적 오류 취소* [arXiv:2201.09866](https://arxiv.org/abs/2201.09866)
## 요구 사항
이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요.

- Qiskit SDK v2.0 이상 ([시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함)
- Qiskit Runtime v0.22 이상 (`pip install qiskit-ibm-runtime`)
## 설정
아래 셀에서 관련 패키지를 가져오고 Backend의 토폴로지를 따르는 2차원 횡자기장 이징 모델의 트로터화된 시간 진화 Circuit을 구성하는 헬퍼 함수를 생성합니다.

In [1]:
from __future__ import annotations
from collections.abc import Sequence
from collections import defaultdict
import numpy as np
import rustworkx
import matplotlib.pyplot as plt

from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.circuit.library import CXGate, CZGate, ECRGate
from qiskit.providers import Backend
from qiskit.visualization import plot_error_map
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import PubResult

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator


"""Trotter circuit generation"""


def remove_qubit_couplings(
    couplings: Sequence[tuple[int, int]], qubits: Sequence[int] | None = None
) -> list[tuple[int, int]]:
    """Remove qubits from a coupling list.

    Args:
        couplings: A sequence of qubit couplings.
        qubits: Optional, the qubits to remove.

    Returns:
        The input couplings with the specified qubits removed.
    """
    if qubits is None:
        return couplings
    qubits = set(qubits)
    return [edge for edge in couplings if not qubits.intersection(edge)]


def coupling_qubits(
    *couplings: Sequence[tuple[int, int]],
    allowed_qubits: Sequence[int] | None = None,
) -> list[int]:
    """Return a sorted list of all qubits involved in one or more couplings lists.

    Args:
        couplings: one or more coupling lists.
        allowed_qubits: Optional, the allowed qubits to include. If None all
            qubits are allowed.

    Returns:
        The intersection of all qubits in the couplings and the allowed qubits.
    """
    qubits = set()
    for edges in couplings:
        for edge in edges:
            qubits.update(edge)
    if allowed_qubits is not None:
        qubits = qubits.intersection(allowed_qubits)
    return list(qubits)


def construct_layer_couplings(
    backend: Backend,
) -> list[list[tuple[int, int]]]:
    """Separate a coupling map into disjoint 2-qubit gate layers.

    Args:
        backend: A backend to construct layer couplings for.

    Returns:
        A list of disjoint layers of directed couplings for the input coupling map.
    """
    coupling_graph = backend.coupling_map.graph.to_undirected(
        multigraph=False
    )
    edge_coloring = rustworkx.graph_bipartite_edge_color(coupling_graph)

    layers = defaultdict(list)
    for edge_idx, color in edge_coloring.items():
        layers[color].append(
            coupling_graph.get_edge_endpoints_by_index(edge_idx)
        )
    layers = [sorted(layers[i]) for i in sorted(layers.keys())]

    return layers


def entangling_layer(
    gate_2q: str,
    couplings: Sequence[tuple[int, int]],
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generating a entangling layer for the specified couplings.

    This corresponds to a Trotter layer for a ZZ Ising term with angle Pi/2.

    Args:
        gate_2q: The 2-qubit basis gate for the layer, should be "cx", "cz", or "ecr".
        couplings: A sequence of qubit couplings to add CX gates to.
        qubits: Optional, the physical qubits for the layer. Any couplings involving
            qubits not in this list will be removed. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The QuantumCircuit for the entangling layer.
    """
    # Get qubits and convert to set to order
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(couplings)))
    qubits = set(qubits)

    # Mapping of physical qubit to virtual qubit
    qubit_mapping = {q: i for i, q in enumerate(qubits)}

    # Convert couplings to indices for virtual qubits
    indices = [
        [qubit_mapping[i] for i in edge]
        for edge in couplings
        if qubits.issuperset(edge)
    ]

    # Layer circuit on virtual qubits
    circuit = QuantumCircuit(len(qubits))

    # Get 2-qubit basis gate and pre and post rotation circuits
    gate2q = None
    pre = QuantumCircuit(2)
    post = QuantumCircuit(2)

    if gate_2q == "cx":
        gate2q = CXGate()
        # Pre-rotation
        pre.sdg(0)
        pre.z(1)
        pre.sx(1)
        pre.s(1)
        # Post-rotation
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "ecr":
        gate2q = ECRGate()
        # Pre-rotation
        pre.z(0)
        pre.s(1)
        pre.sx(1)
        pre.s(1)
        # Post-rotation
        post.x(0)
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "cz":
        gate2q = CZGate()
        # Identity pre-rotation
        # Post-rotation
        post.sdg([0, 1])
    else:
        raise ValueError(
            f"Invalid 2-qubit basis gate {gate_2q}, should be 'cx', 'cz', or 'ecr'"
        )

    # Add 1Q pre-rotations
    for inds in indices:
        circuit.compose(pre, qubits=inds, inplace=True)

    # Use barriers around 2-qubit basis gate to specify a layer for PEA noise learning
    circuit.barrier()
    for inds in indices:
        circuit.append(gate2q, (inds[0], inds[1]))
    circuit.barrier()

    # Add 1Q post-rotations after barrier
    for inds in indices:
        circuit.compose(post, qubits=inds, inplace=True)

    # Add physical qubits as metadata
    circuit.metadata["physical_qubits"] = tuple(qubits)

    return circuit


def trotter_circuit(
    theta: Parameter | float,
    layer_couplings: Sequence[Sequence[tuple[int, int]]],
    num_steps: int,
    gate_2q: str | None = "cx",
    backend: Backend | None = None,
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generate a Trotter circuit for the 2D Ising

    Args:
        theta: The angle parameter for X.
        layer_couplings: A list of couplings for each entangling layer.
        num_steps: the number of Trotter steps.
        gate_2q: The 2-qubit basis gate to use in entangling layers.
            Can be "cx", "cz", "ecr", or None if a backend is provided.
        backend: A backend to get the 2-qubit basis gate from, if provided
            will override the basis_gate field.
        qubits: Optional, the allowed physical qubits to truncate the
            couplings to. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The Trotter circuit.
    """
    if backend is not None:
        try:
            basis_gates = backend.configuration().basis_gates
        except AttributeError:
            basis_gates = backend.basis_gates
        for gate in ["cx", "cz", "ecr"]:
            if gate in basis_gates:
                gate_2q = gate
                break

    # If no qubits, get the largest qubit from all layers and
    # specify the range so the same one is used for all layers.
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(layer_couplings)))

    # Generate the entangling layers
    layers = [
        entangling_layer(gate_2q, couplings, qubits=qubits)
        for couplings in layer_couplings
    ]

    # Construct the circuit for a single Trotter step
    num_qubits = len(qubits)
    trotter_step = QuantumCircuit(num_qubits)
    trotter_step.rx(theta, range(num_qubits))
    for layer in layers:
        trotter_step.compose(layer, range(num_qubits), inplace=True)

    # Construct the circuit for the specified number of Trotter steps
    circuit = QuantumCircuit(num_qubits)
    for _ in range(num_steps):
        circuit.rx(theta, range(num_qubits))
        for layer in layers:
            circuit.compose(layer, range(num_qubits), inplace=True)

    circuit.metadata["physical_qubits"] = tuple(qubits)
    return circuit


"""Result visualization functions"""


def plot_trotter_results(
    pub_result: PubResult,
    angles: Sequence[float],
    plot_noise_factors: Sequence[float] | None = None,
    plot_extrapolator: Sequence[str] | None = None,
    exact: np.ndarray = None,
    close: bool = True,
):
    """Plot average magnetization from ZNE result data.
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx angle values for the experiment.
        plot_raw: If provided plot the unextrapolated data for the noise factors.
        plot_extrapolator: If provided plot all extrapolators, if False only plot
            the Automatic method.
        exact: Optional, the exact values to include in the plot. Should be a 1D
            array-like where the values represent exact magnetization.
        close: Close the Matplotlib figure before returning.
    Returns:
        The figure.
    """
    data = pub_result.data

    evs = data.evs
    num_qubits = evs.shape[0]
    num_params = evs.shape[1]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # Take average magnetization of qubits and its standard error
    x_vals = angles / np.pi
    y_vals = np.mean(evs, axis=0)
    y_errs = np.std(evs, axis=0) / np.sqrt(num_qubits)

    fig, _ = plt.subplots(1, 1)

    # Plot auto method
    plt.errorbar(x_vals, y_vals, y_errs, fmt="o-", label="ZNE (automatic)")

    # Plot individual extrapolator results
    if plot_extrapolator:
        y_vals_extrap = np.mean(data.evs_extrapolated, axis=0)
        y_errs_extrap = np.std(data.evs_extrapolated, axis=0) / np.sqrt(
            num_qubits
        )
        for i, extrap in enumerate(plot_extrapolator):
            plt.errorbar(
                x_vals,
                y_vals_extrap[:, i, 0],
                y_errs_extrap[:, i, 0],
                fmt="s-.",
                alpha=0.5,
                label=f"ZNE ({extrap})",
            )

    # Plot raw results
    if plot_noise_factors:
        y_vals_raw = np.mean(data.evs_noise_factors, axis=0)
        y_errs_raw = np.std(data.evs_noise_factors, axis=0) / np.sqrt(
            num_qubits
        )
        for i, nf in enumerate(plot_noise_factors):
            plt.errorbar(
                x_vals,
                y_vals_raw[:, i],
                y_errs_raw[:, i],
                fmt="d:",
                alpha=0.5,
                label=f"Raw (nf={nf:.1f})",
            )

    # Plot exact data
    if exact is not None:
        plt.plot(x_vals, exact, "--", color="black", alpha=0.5, label="Exact")

    plt.ylim(-0.1, 1.2)
    plt.xlabel("θ/π")
    plt.ylabel(r"$\overline{\langle Z \rangle}$")
    plt.legend()
    plt.title(
        f"Error Mitigated Average Magnetization for Rx(θ) [{num_qubits}-qubit]"
    )
    if close:
        plt.close(fig)
    return fig


def plot_qubit_zne_data(
    pub_result: PubResult,
    angles: Sequence[float],
    qubit: int,
    noise_factors: Sequence[float],
    extrapolator: Sequence[str] | None = None,
    extrapolated_noise_factors: Sequence[float] | None = None,
    num_cols: int | None = None,
    close: bool = True,
):
    """Plot ZNE extrapolation data for specific virtual qubit
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx theta angles used for the experiment.
        qubit: The virtual qubit index to plot.
        noise_factors: the raw noise factors.
        extrapolator: The extrapolator metadata for multiple extrapolators.
        extrapolated_noise_factors: The noise factors used for extrapolation.
        num_cols: The number of columns for the generated subplots.
        close: Close the Matplotlib figure before returning.
    Returns:
        The Matplotlib figure.
    """
    data = pub_result.data

    evs_auto = data.evs[qubit]
    stds_auto = data.stds[qubit]
    evs_extrap = data.evs_extrapolated[qubit]
    stds_extrap = data.stds_extrapolated[qubit]
    evs_raw = data.evs_noise_factors[qubit]
    stds_raw = data.stds_noise_factors[qubit]

    num_params = evs_auto.shape[0]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # Make a square subplot
    num_cols = num_cols or int(np.ceil(np.sqrt(num_params)))
    num_rows = int(np.ceil(num_params / num_cols))
    fig, axes = plt.subplots(
        num_rows, num_cols, sharex=True, sharey=True, figsize=(12, 5)
    )
    fig.suptitle(f"ZNE data for virtual qubit {qubit}")

    for pidx, ax in zip(range(num_params), axes.flat):
        # Plot auto extrapolated
        ax.errorbar(
            0,
            evs_auto[pidx],
            stds_auto[pidx],
            fmt="o",
            label="PEA (automatic)",
        )

        # Plot extrapolators
        if (
            extrapolator is not None
            and extrapolated_noise_factors is not None
        ):
            for i, method in enumerate(extrapolator):
                ax.errorbar(
                    extrapolated_noise_factors,
                    evs_extrap[pidx, i],
                    stds_extrap[pidx, i],
                    fmt="-",
                    alpha=0.5,
                    label=f"PEA ({method})",
                )

        # Plot raw
        ax.errorbar(
            noise_factors, evs_raw[pidx], stds_raw[pidx], fmt="d", label="Raw"
        )

        ax.set_yticks([0, 0.5, 1, 1.5, 2])
        ax.set_ylim(0, max(1, 1.1 * max(evs_auto)))

        ax.set_xticks([0, *noise_factors])
        ax.set_title(f"θ/π = {angles[pidx]/np.pi:.2f}")
        if pidx == 0:
            ax.set_ylabel(r"$\langle Z_{" + str(qubit) + r"} \rangle$")
        if pidx == num_params - 1:
            ax.set_xlabel("Noise Factor")
            ax.legend()
    plt.tight_layout()
    if close:
        plt.close(fig)
    return fig

## 소규모 시뮬레이터 예제
런타임 오류 완화는 시뮬레이터에서 지원되지 않으므로 이 단계를 건너뜁니다.

## 대규모 하드웨어 예제
### 1단계: 고전적 입력을 양자 문제로 매핑
#### 매개변수화된 이징 모델 Circuit 생성
##### Backend 설정
먼저, 실행할 Backend를 선택합니다. 이 데모는 127-Qubit Backend에서 실행되지만, 이용 가능한 다른 Backend로 변경할 수 있습니다.

In [2]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend

<IBMBackend('ibm_fez')>

##### Define entangling layer couplings
To implement the Trotterized Ising simulation, define three layers of two-qubit gate couplings for the device, to be repeated at each of the Trotter steps. These define the three twirled layers you need to learn the noise for to implement mitigation.

In [3]:
layer_couplings = construct_layer_couplings(backend)
for i, layer in enumerate(layer_couplings):
    print(f"Layer {i}:\n{layer}\n")

Layer 0:
[(2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 23), (18, 31), (19, 35), (20, 21), (25, 37), (26, 27), (28, 29), (33, 39), (36, 41), (38, 49), (42, 43), (45, 46), (47, 57), (51, 52), (53, 54), (56, 63), (58, 71), (59, 75), (61, 62), (64, 65), (66, 67), (68, 69), (72, 73), (76, 81), (79, 93), (82, 83), (84, 85), (86, 87), (88, 89), (91, 98), (94, 95), (97, 107), (99, 115), (100, 101), (102, 103), (105, 117), (108, 109), (110, 111), (113, 114), (116, 121), (118, 129), (123, 136), (124, 125), (126, 127), (130, 131), (132, 133), (135, 139), (138, 151), (142, 143), (144, 145), (146, 147), (152, 153), (154, 155)]

Layer 1:
[(0, 1), (3, 16), (5, 6), (7, 8), (11, 18), (13, 14), (17, 27), (21, 22), (23, 24), (25, 26), (29, 38), (30, 31), (32, 33), (34, 35), (39, 53), (41, 42), (43, 56), (44, 45), (47, 48), (49, 50), (51, 58), (54, 55), (57, 67), (60, 61), (62, 63), (65, 66), (69, 78), (70, 71), (73, 79), (74, 75), (77, 85), (80, 81), (83, 84), (87, 97), (89, 90), (9

##### Define entangling layer couplings
Trotterized Ising 시뮬레이션을 구현하기 위해 디바이스에서 각 Trotter 단계마다 반복될 2-큐비트 게이트 커플링의 세 레이어를 정의합니다. 이는 노이즈를 학습하여 완화를 구현하는 데 필요한 세 가지 트월링된 레이어를 정의합니다.

In [4]:
# Plot gate error map
# NOTE: These can change over time, so your results may look different
plot_error_map(backend)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/fccef708-0.avif" alt="Output of the previous code cell" />

In [5]:
bad_qubits = {
    32,
    33,
    71,
    72,
    73,
    102,
    103,
}  # qubits removed based on high coupling error (1.00)
good_qubits = list(set(range(backend.num_qubits)).difference(bad_qubits))
print("Physical qubits:\n", good_qubits)

Physical qubits:
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]


##### Remove bad qubits
Backend의 커플링 맵을 살펴보고 높은 오류를 가진 커플링에 연결된 큐비트가 있는지 확인합니다. 이러한 "불량" 큐비트를 실험에서 제거합니다.

In [6]:
num_steps = 6
theta = Parameter("theta")
circuit = trotter_circuit(
    theta, layer_couplings, num_steps, qubits=good_qubits, backend=backend
)

#### Create a list of parameter values to be assigned later

In [7]:
num_params = 12

# 12 parameter values for Rx between [0, pi/2].
# Reshape to outer product broadcast with observables
parameter_values = np.linspace(0, np.pi / 2, num_params).reshape(
    (num_params, 1)
)
num_params = parameter_values.size

### Step 2: Optimize problem for quantum hardware execution

#### ISA circuit
Before running the circuit on hardware, optimize it for hardware execution. This process involves a few steps:

* Pick a qubit layout that maps the virtual qubits of your circuit to physical qubits on the hardware.
* Insert swap gates as needed to route interactions between qubits that are not connected.
* Translate the gates in our circuit to [Instruction Set Architecture (ISA)](/docs/guides/transpile#instruction-set-architecture) instructions that can directly be executed on the hardware.
* Perform circuit optimizations to minimize the circuit depth and gate count.

Although the transpiler built into Qiskit can perform all of these steps, this tutorial demonstrates building the utility-scale Trotter circuit in a ground-up fashion. Select the good physical qubits and define entangling layers on connected qubit pairs from those selected qubits. Nonetheless, you still need to translate non-ISA gates in the circuit and avail any circuit optimization offered by the transpiler.

Transpile your circuit for the chosen backend by creating a pass manager and then running the pass manager on the circuit. Also, fix the initial layout of the circuit to the already selected `good_qubits`. An easy way to create a pass manager is to use the [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) function. See [Transpile with pass managers](/docs/guides/transpile-with-pass-managers) for a more detailed explanation of transpiling with pass managers.

In [8]:
pm = generate_preset_pass_manager(
    backend=backend,
    initial_layout=good_qubits,
    layout_method="trivial",
    optimization_level=1,
)

isa_circuit = pm.run(circuit)

##### Main Trotter circuit generation

In [9]:
observables = []
num_qubits = len(good_qubits)
for q in range(num_qubits):
    observables.append(
        SparsePauliOp("I" * (num_qubits - q - 1) + "Z" + "I" * q)
    )

#### Create a list of parameter values to be assigned later

In [10]:
isa_observables = [
    [obs.apply_layout(layout=isa_circuit.layout)] for obs in observables
]

### 2단계: 양자 하드웨어 실행을 위한 문제 최적화
#### ISA circuit
회로를 하드웨어에서 실행하기 전에 하드웨어 실행에 맞게 최적화합니다. 이 과정에는 몇 가지 단계가 포함됩니다:

* 회로의 가상 큐비트를 하드웨어의 물리적 큐비트에 매핑하는 큐비트 레이아웃을 선택합니다.
* 연결되지 않은 큐비트 간의 상호 작용을 라우팅하기 위해 필요한 경우 스왑 게이트를 삽입합니다.
* 회로의 게이트를 하드웨어에서 직접 실행할 수 있는 [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) 명령어로 변환합니다.
* 회로 깊이와 게이트 수를 최소화하기 위한 회로 최적화를 수행합니다.

Qiskit에 내장된 Transpiler가 이 모든 단계를 수행할 수 있지만, 이 튜토리얼에서는 유틸리티 규모의 Trotter 회로를 처음부터 구축하는 방법을 보여줍니다. 좋은 물리적 큐비트를 선택하고 선택한 큐비트 중 연결된 큐비트 쌍에 엔탱글링 레이어를 정의합니다. 그럼에도 불구하고 회로에서 비-ISA 게이트를 변환하고 Transpiler가 제공하는 회로 최적화를 활용해야 합니다.

패스 매니저를 생성한 후 회로에 실행하여 선택한 Backend에 맞게 회로를 트랜스파일합니다. 또한, 이미 선택한 `good_qubits`에 회로의 초기 레이아웃을 고정합니다. 패스 매니저를 쉽게 생성하는 방법은 [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) 함수를 사용하는 것입니다. 패스 매니저를 사용한 트랜스파일에 대한 더 자세한 설명은 [Transpile with pass managers](/guides/transpile-with-pass-managers)를 참고하세요.

In [11]:
pub = (isa_circuit, isa_observables, parameter_values)

#### ISA observables
다음으로, 필요한 수의 $\langle I \rangle$ 항을 채워 각 가상 큐비트에 대한 모든 weight-1 $\langle Z \rangle$ 관측량을 생성합니다.

In [12]:
# Experiment options
num_randomizations = 700
num_randomizations_learning = 40
max_batch_circuits = 3 * num_params
shots_per_randomization = 64
learning_pair_depths = [0, 1, 2, 4, 6, 12, 24]
noise_factors = [1, 1.3, 1.6]
extrapolated_noise_factors = np.linspace(0, max(noise_factors), 20)

# Base option formatting
options = {
    # Builtin resilience settings for ZNE
    "resilience": {
        "measure_mitigation": True,
        "zne_mitigation": True,
        # TREX noise learning configuration
        "measure_noise_learning": {
            "num_randomizations": num_randomizations_learning,
            "shots_per_randomization": 1024,
        },
        # PEA noise model configuration
        "layer_noise_learning": {
            "max_layers_to_learn": 3,
            "layer_pair_depths": learning_pair_depths,
            "shots_per_randomization": shots_per_randomization,
            "num_randomizations": num_randomizations_learning,
        },
        "zne": {
            "amplifier": "pea",
            "noise_factors": noise_factors,
            "extrapolator": ("exponential", "linear"),
            "extrapolated_noise_factors": extrapolated_noise_factors.tolist(),
        },
    },
    # Randomization configuration
    "twirling": {
        "num_randomizations": num_randomizations,
        "shots_per_randomization": shots_per_randomization,
        "strategy": "active-circuit",
    },
    # Optional Dynamical Decoupling (DD)
    "dynamical_decoupling": {"enable": True, "sequence_type": "XY4"},
    # Job tag
    "environment": {"job_tags": ["TUT_PEA"]},
}

트랜스파일 과정에서 회로의 가상 큐비트가 하드웨어의 물리적 큐비트에 매핑되었습니다. 큐비트 레이아웃에 관한 정보는 트랜스파일된 회로의 `layout` 속성에 저장됩니다. 관측량도 가상 큐비트로 정의되어 있으므로 이 레이아웃을 관측량에 적용해야 합니다. 이는 `SparsePauliOp`의 `apply_layout` 메서드를 사용하여 수행됩니다.

다음 코드 블록에서 각 관측량이 목록으로 감싸져 있다는 점에 유의하세요. 이는 파라미터 값과 *브로드캐스트*하여 각 큐비트 관측량이 각 theta 값에 대해 측정되도록 하기 위해 수행됩니다. 프리미티브의 브로드캐스팅 규칙은 [프리미티브 문서](/guides/primitives)에서 확인할 수 있습니다.

In [13]:
estimator = Estimator(mode=backend, options=options)
job = estimator.run([pub])
print(f"Job ID {job.job_id()}")

Job ID d7fa8oe2cugc739qbb10


In [14]:
job.status()

'DONE'

#### Configure Estimator options
다음으로 완화 실험을 실행하는 데 필요한 `Estimator` 옵션을 구성합니다. 여기에는 엔탱글링 레이어의 노이즈 학습 옵션과 ZNE 외삽 옵션이 포함됩니다.

다음 구성을 사용합니다:

In [15]:
primitive_result = job.result()

##### Explanation of ZNE options
다음은 실험 브랜치의 추가 옵션에 대한 세부 사항입니다. 이러한 옵션과 이름은 확정되지 않았으며, 공식 릴리스 전에 변경될 수 있습니다.

* **amplifier**: 의도한 노이즈 계수로 노이즈를 증폭할 때 사용하는 방법입니다.
  허용되는 값은 2-큐비트 기저 게이트를 반복하여 증폭하는 `"gate_folding"`,
  그리고 트월링된 2-큐비트 기저 게이트 레이어에 대한 Pauli-트월링된 노이즈 모델을 학습한 후 확률론적 샘플링으로 증폭하는 `"pea"`입니다. 추가 옵션으로 `"gate_folding_front"` 및 `"gate_folding_back"`이 있으며, 이는 [API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options#amplifier)에서 설명됩니다.
* **extrapolated\_noise\_factors**: 외삽 모델을 평가할 하나 이상의 노이즈 계수 값을 지정합니다. 값의 시퀀스를 지정하면 반환된 결과는 외삽 모델에 대해 지정된 노이즈 계수로 배열 값이 됩니다. 값
  0은 제로 노이즈 외삽에 해당합니다.

#### Run the experiment

In [16]:
pub_result = primitive_result[0]

print(
    f"{pub_result.data.evs.shape=}\n"
    f"{pub_result.data.evs_extrapolated.shape=}\n"
    f"{pub_result.data.evs_noise_factors.shape=}\n"
)

pub_result.data.evs.shape=(149, 12)
pub_result.data.evs_extrapolated.shape=(149, 12, 2, 20)
pub_result.data.evs_noise_factors.shape=(149, 12, 3)



Several metadata fields are also available in the `PrimitiveResult`. The metadata includes

* `resilience/zne/noise_factors`: The raw noise factors
* `resilience/zne/extrapolator`: The extrapolators used for each result

In [17]:
primitive_result.metadata

{'dynamical_decoupling': {'enable': True,
  'sequence_type': 'XY4',
  'extra_slack_distribution': 'middle',
  'scheduling_method': 'alap'},
 'twirling': {'enable_gates': True,
  'enable_measure': True,
  'num_randomizations': 700,
  'shots_per_randomization': 64,
  'interleave_randomizations': True,
  'strategy': 'active-circuit'},
 'resilience': {'measure_mitigation': True,
  'zne_mitigation': True,
  'pec_mitigation': False,
  'zne': {'noise_factors': [1.0, 1.3, 1.6],
   'extrapolator': ['exponential', 'linear'],
   'extrapolated_noise_factors': [0.0,
    0.08421052631578947,
    0.16842105263157894,
    0.25263157894736843,
    0.3368421052631579,
    0.42105263157894735,
    0.5052631578947369,
    0.5894736842105263,
    0.6736842105263158,
    0.7578947368421053,
    0.8421052631578947,
    0.9263157894736842,
    1.0105263157894737,
    1.0947368421052632,
    1.1789473684210525,
    1.263157894736842,
    1.3473684210526315,
    1.431578947368421,
    1.5157894736842106,
    1.

The `PubResult` object has additional resilience metadata about the learned noise models used in mitigation.

In [18]:
# Print learned layer noise metadata
for field, value in pub_result.metadata["resilience"]["layer_noise"].items():
    print(f"{field}: {value}")

noise_overhead: 9.2584227461744e+229
total_mitigated_layers: 18
unique_mitigated_layers: 3
unique_mitigated_layers_noise_overhead: [2.0713004613510885e+36, 10.600275591731494, 9.687147432958504]


In [ ]:
# Exact data computed using the methods described in the original reference
# Y. Kim et al. "Evidence for the utility of quantum computing before fault tolerance" (Nature 618,
# 500–505 (2023)) Directly used here for brevity
exact_data = np.array(
    [
        1,
        0.9899,
        0.9531,
        0.8809,
        0.7536,
        0.5677,
        0.3545,
        0.1607,
        0.0539,
        0.0103,
        0.0012,
        0.0,
    ]
)

### 4단계: 후처리 및 원하는 고전 형식으로 결과 반환
실험이 완료되면 결과를 확인할 수 있습니다. 원시(raw) 기댓값과 완화된(mitigated) 기댓값을 가져와 정확한 결과와 비교합니다. 그런 다음 각 매개변수에 대해 모든 Qubit에 걸쳐 평균한 완화된(외삽된) 기댓값과 원시 기댓값을 플롯합니다. 마지막으로 선택한 개별 Qubit에 대한 기댓값을 플롯합니다.

In [20]:
zne_metadata = primitive_result.metadata["resilience"]["zne"]
# Plot Trotter simulation results
fig = plot_trotter_results(
    pub_result,
    parameter_values,
    plot_extrapolator=zne_metadata["extrapolator"],
    plot_noise_factors=zne_metadata["noise_factors"],
    exact=exact_data,
)
display(fig)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/e466736a-0.avif" alt="Output of the previous code cell" />

#### 일반적인 결과 형태와 메타데이터
`PrimitiveResult` 객체에는 `PubResult`라는 리스트 형태의 구조가 포함되어 있습니다. Estimator에 하나의 PUB만 제출했기 때문에 `PrimitiveResult`에는 단일 `PubResult` 객체가 포함됩니다.

PUB(primitive unified bloc) 결과의 기댓값과 표준 오차는 배열 형태입니다. ZNE를 사용하는 Estimator 작업의 경우 `PubResult`의 `DataBin` 컨테이너에서 여러 기댓값 및 표준 오차 데이터 필드를 사용할 수 있습니다. 여기서는 기댓값에 대한 데이터 필드를 간략히 설명합니다(표준 오차(`stds`)에도 유사한 데이터 필드가 있습니다).

1. `pub_result.data.evs`: 영 잡음(휴리스틱적으로 최적의 외삽 기반)에 해당하는 기댓값입니다.
    - 첫 번째 축은 관측값 $\langle Z_i\rangle$에 대한 가상 qubit 인덱스입니다 ($124$개의 가상 qubit/관측값)
    - 두 번째 축은 $\theta$에 대한 매개변수 값을 인덱싱합니다 ($12$개의 매개변수 값)
2. `pub_result.data.evs_extrapolated`: 모든 외삽기(extrapolator)에 대한 외삽된 잡음 인수의 기댓값입니다. 이 배열에는 두 개의 추가 축이 있습니다.
    - 세 번째 축은 외삽 방법을 인덱싱합니다 ($2$개의 외삽기: `exponential` 및 `linear`)
    - 마지막 축은 `extrapolated_noise_factors`를 인덱싱합니다 (옵션에서 지정한 $20$개의 외삽 포인트)
3. `pub_result.data.evs_noise_factors`: 각 잡음 인수에 대한 원시 기댓값입니다.
   - 세 번째 축은 원시 `noise_factors`를 인덱싱합니다 ($3$개의 인수)

In [21]:
virtual_qubit = 1
plot_qubit_zne_data(
    pub_result=pub_result,
    angles=parameter_values,
    qubit=virtual_qubit,
    noise_factors=zne_metadata["noise_factors"],
    extrapolator=zne_metadata["extrapolator"],
    extrapolated_noise_factors=zne_metadata["extrapolated_noise_factors"],
)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/bea9695a-0.avif" alt="Output of the previous code cell" />

## Next steps
<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- A [tutorial](/docs/tutorials/combine-error-mitigation-techniques) focused on combining error mitigation techniques.
- Detailed [documentation](/docs/guides/error-mitigation-and-suppression-techniques) on the error mitigation techniques available in Qiskit.
- Additional lessons covering utility-scale experiments: [Utility II](/learning/courses/utility-scale-quantum-computing/utility-ii) and [Utility III](/learning/courses/utility-scale-quantum-computing/utility-iii).
</Admonition>